In [ ]:
# -*- coding: utf-8 -*-
"""factor_divergence_orderlayer_gate_v1: 指标分歧高A引擎 × 委托笔数远档门 (D1)

通道: factorlib 技术指标分歧 (indicator_divergence 验证 A=0.799) × bar1m 委托笔数 2-5 档门
      (硬编码表名; 无 pivot/lstsq/apply/rolling; 日频 shift(1) 先于 merge)

mechanism (金融机制)
====================
A 引擎: 技术指标分歧度 → 反转调制 (indicator_divergence 核心, 官方 A=0.799/B=0.867):
  指标一致 = 趋势有资金共识; 分歧 = 趋势脆弱 → A股短线反转占优。
门: 委托笔数远档占比 (bid/ask_num_orders2-5 全场唯一, rlsr_v2 run-error 未占住):
  远档委托笔数占比高 = 深度挂单厚实(机构护盘/对倒) → 反转方向确认;
  仅一档拥挤 = 散户短线博弈 → 反转信号弱。
  用前日远档占比 (shift(1)) 做门, 与当日分歧方向对齐时放大。

novelty (与现有因子差异)
========================
- indicator_divergence/segment: A 引擎同源, 但门不同 (segment 用量能熵; 本因子用委托笔数 2-5 档)
- trade_granularity_v3/reversal_opposition: 用委托笔数 1 档 (bid/ask_num_orders1) 做对侧墙;
  本因子用 **2-5 档远档占比** (bid_num_orders2-5/ask_num_orders2-5, 池内零占用)
- rlsr_ocs_v2: 用了 2-5 档但含 pivot_table → run-error 未占住通道

安全设计 (对齐已验证通过模板)
============================
  factorlib 硬编码表名; bar1m 用 {datasources["bar1m"]}
  委托笔数 → 日频 groupby.agg(sum) → 日频 shift(1) 按 instrument, 先于 merge
  无 .apply/lstsq/agg-lambda/pivot/rolling/cumsum/分钟 shift; 输出恰好 3 列
"""

import numpy as np
import pandas as pd

try:
    import dai
except Exception:
    dai = None


def main(datasources, start_date, end_date):
    BUFFER_DAYS = 30
    query_start = (
        pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)
    ).strftime("%Y-%m-%d %H:%M:%S")

    # ---- 1. bar1m 委托笔数 → 日频 (顺序无关聚合) ----
    sql_bar = f"""
        SELECT date, instrument,
            bid_num_orders1, bid_num_orders2, bid_num_orders3,
            bid_num_orders4, bid_num_orders5,
            ask_num_orders1, ask_num_orders2, ask_num_orders3,
            ask_num_orders4, ask_num_orders5
        FROM {datasources["bar1m"]}
        WHERE close > 0
    """
    bar = dai.query(sql_bar, filters={"date": [query_start, end_date]}, compression=True).df()
    if bar.empty:
        raise ValueError("Empty bar1m data")
    bar["date"] = pd.to_datetime(bar["date"])
    bar["trading_day"] = bar["date"].dt.strftime("%Y-%m-%d")

    ord_cols = [f"{s}_num_orders{i}" for s in ("bid", "ask") for i in range(1, 6)]
    for c in ord_cols:
        bar[c] = pd.to_numeric(bar[c], errors="coerce").fillna(0.0)

    ord_daily = bar.groupby(["trading_day", "instrument"], sort=False)[ord_cols].sum().reset_index()
    ord_daily = ord_daily.rename(columns={"trading_day": "date"})
    ord_daily["date"] = pd.to_datetime(ord_daily["date"])

    # 远档占比: (2+3+4+5) / (1+2+3+4+5), 买侧与卖侧
    b_all = (ord_daily["bid_num_orders1"] + ord_daily["bid_num_orders2"]
             + ord_daily["bid_num_orders3"] + ord_daily["bid_num_orders4"]
             + ord_daily["bid_num_orders5"])
    a_all = (ord_daily["ask_num_orders1"] + ord_daily["ask_num_orders2"]
             + ord_daily["ask_num_orders3"] + ord_daily["ask_num_orders4"]
             + ord_daily["ask_num_orders5"])
    b_far = (ord_daily["bid_num_orders2"] + ord_daily["bid_num_orders3"]
             + ord_daily["bid_num_orders4"] + ord_daily["bid_num_orders5"])
    a_far = (ord_daily["ask_num_orders2"] + ord_daily["ask_num_orders3"]
             + ord_daily["ask_num_orders4"] + ord_daily["ask_num_orders5"])
    ord_daily["far_share_b"] = b_far / (b_all + 1e-8)
    ord_daily["far_share_a"] = a_far / (a_all + 1e-8)
    ord_daily["far_share"] = (ord_daily["far_share_b"] + ord_daily["far_share_a"]) / 2.0

    # T-1 远档占比 (日频 shift 先于 merge)
    ord_daily = ord_daily.sort_values(["instrument", "date"]).reset_index(drop=True)
    ord_daily["prev_far_share"] = ord_daily.groupby("instrument", sort=False)["far_share"].shift(1)

    # ---- 2. factorlib 技术指标 (硬编码表名) ----
    sql_flib = """
        SELECT date, instrument,
            momentum_5, reversal_5, volatility_5,
            macd_diff_12_26_9, macd_dea_12_26_9, macd_hist_12_26_9,
            rsi_12, kdj_k_9_3_3, kdj_d_9_3_3,
            bias_20, cci_14, atr_14
        FROM bigalpha_2026_factorlib
    """
    flib = dai.query(sql_flib, filters={"date": [query_start, end_date]}, compression=True).df()
    if flib.empty:
        raise ValueError("Empty factorlib data")
    flib["date"] = pd.to_datetime(flib["date"])

    daily = pd.merge(
        ord_daily[["date", "instrument", "prev_far_share"]], flib,
        how="inner", on=["date", "instrument"],
    )
    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)

    # ---- 3. 股票池对齐 (merge 后只 rank, 绝不 shift) ----
    stock_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    if stock_pool.empty:
        raise ValueError("Empty instruments data")
    stock_pool["date"] = pd.to_datetime(stock_pool["date"])
    daily = pd.merge(stock_pool, daily, how="inner", on=["date", "instrument"])
    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)

    # ---- 4. 截面算子 (groupby("date"), 只 rank/zscore) ----
    def rk(series):
        return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).groupby(
            daily["date"], sort=False
        ).rank(pct=True).fillna(0.5)

    def cz(series):
        return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).groupby(
            daily["date"], sort=False
        ).transform(lambda x: (x - x.mean()) / (x.std() + 1e-8)).fillna(0.0)

    # ---- 5. A 引擎: 技术指标分歧 (indicator_divergence 结构) ----
    mom = rk(daily["momentum_5"].fillna(0))
    rev = rk(daily["reversal_5"].fillna(0))
    bias = rk(daily["bias_20"].fillna(0))
    macd = rk(daily["macd_diff_12_26_9"].fillna(0) - daily["macd_dea_12_26_9"].fillna(0))
    rsi = rk(daily["rsi_12"].fillna(50))
    kdj = rk(daily["kdj_k_9_3_3"].fillna(0) - daily["kdj_d_9_3_3"].fillna(0))
    cci = rk(daily["cci_14"].fillna(0))

    d_mom = (mom > 0.5).astype(float)
    d_rev = (rev > 0.5).astype(float)
    d_bias = (bias > 0.5).astype(float)
    d_macd = (macd > 0.5).astype(float)
    d_rsi = (rsi > 0.5).astype(float)
    d_kdj = (kdj > 0.5).astype(float)
    d_cci = (cci > 0.5).astype(float)

    ref = 2.0 * d_mom - 1.0
    agree = (
        (d_rev == d_mom).astype(float) + (d_bias == d_mom).astype(float)
        + (d_macd == d_mom).astype(float) + (d_rsi == d_mom).astype(float)
        + (d_kdj == d_mom).astype(float) + (d_cci == d_mom).astype(float)
    ) / 6.0
    div = 1.0 - agree

    t1 = -ref * div
    stretch = rk(daily["bias_20"].abs().fillna(0))
    tight = 1.0 - rk(daily["atr_14"].fillna(0))
    extreme = stretch * tight
    ref_bias = 2.0 * d_bias - 1.0
    t2 = -ref_bias * extreme
    overbought = (daily["rsi_12"].fillna(50) > 70).astype(float)
    oversold = (daily["rsi_12"].fillna(50) < 30).astype(float)
    exhaust = (d_macd * overbought) - ((1.0 - d_macd) * oversold)
    t3 = -exhaust
    consensus_up = d_kdj * d_bias
    consensus_down = (1.0 - d_kdj) * (1.0 - d_bias)
    t4 = -consensus_up + consensus_down

    engine_raw = 0.40 * t1 + 0.25 * t2 + 0.20 * t3 + 0.15 * t4
    engine_raw = engine_raw.replace([np.inf, -np.inf], np.nan)
    if engine_raw.nunique(dropna=True) <= 1:
        engine_raw = -ref
    engine_z = cz(engine_raw)

    # ---- 6. B 门: 委托笔数远档占比 (T-1, 池内零占用) ----
    gate = cz(daily["prev_far_share"].fillna(0.5))

    # ---- 7. 条件乘积融合 (保 B) ----
    gate_z = gate.clip(-3, 3)
    raw = engine_z * (0.5 + 0.5 * np.maximum(0, engine_z * gate_z))
    raw = pd.to_numeric(raw, errors="coerce").replace([np.inf, -np.inf], np.nan)

    # 退化保护
    if raw.nunique(dropna=True) <= 1:
        raw = engine_z

    # ---- 8. 截面 rank 输出 (恰好 3 列) ----
    daily["factor"] = raw.groupby(daily["date"], sort=False).rank(pct=True).fillna(0.5)
    result = daily[["date", "instrument", "factor"]].copy()
    result["date"] = pd.to_datetime(result["date"])
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    result = result.dropna(subset=["factor"])
    return result[
        (result["date"] >= pd.to_datetime(start_date))
        & (result["date"] <= pd.to_datetime(end_date))
    ]
